# BlueOrbit Office Assistant — RAG Pipeline

## Objective

This notebook implements the Retrieval-Augmented Generation (RAG) component
of the BlueOrbit Office Assistant.

The RAG pipeline allows the assistant to answer questions about
BlueOrbit Solutions Pvt. Ltd. policies and company information stored
in PDF documents.

### Pipeline

PDF Documents
↓
Text Extraction
↓
Text Cleaning
↓
Text Chunking
↓
Embedding Generation
↓
FAISS Vector Database
↓
Semantic Retrieval
↓
Gemini LLM
↓
Final Answer

## 1. Import Required Libraries

We use:

- `pypdf` → extract text from PDF files
- `numpy` → numerical operations
- `faiss` → vector similarity search
- `google-genai` → Gemini embeddings and LLM
- `python-dotenv` → load the Gemini API key securely
- `pathlib` → handle file paths

In [1]:
from pathlib import Path

import numpy as np
import faiss

from pypdf import PdfReader
from dotenv import load_dotenv

from google import genai

## 2. Project Paths

The PDFs are stored inside:

`data/pdfs/`

Since this notebook is inside the `rag/` folder, we move one level up
to reach the project root.

In [2]:
BASE_DIR = Path.cwd().parent
PDF_DIR = BASE_DIR / "data" / "pdfs"

print("Project directory:", BASE_DIR)
print("PDF directory:", PDF_DIR)

Project directory: /home/nineleaps/BlueOrbit-Office-Assistant
PDF directory: /home/nineleaps/BlueOrbit-Office-Assistant/data/pdfs


In [3]:
pdf_files = list(PDF_DIR.glob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)

Number of PDFs: 5
- Work_From_Home_Policy.pdf
- Employee_Handbook.pdf
- Security_Policy.pdf
- Leave_Policy.pdf
- Travel_Policy.pdf


## 3. Configure Gemini API

The Gemini API key should not be hardcoded in the notebook.

Instead, we store it in a `.env` file and load it using `python-dotenv`.



In [4]:
load_dotenv()

import os

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found. Check your .env file.")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


## 4. PDF Text Extraction

LLMs cannot directly search our PDF files.

First, we extract the text from every PDF page.

For each document we store:

- `source` → PDF filename
- `text` → extracted document text

This source information will later help us identify where an answer
came from.

In [5]:
def load_pdfs(pdf_directory):
    documents = []

    for pdf_path in pdf_directory.glob("*.pdf"):
        reader = PdfReader(pdf_path)

        text = ""

        for page in reader.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

        documents.append({
            "source": pdf_path.name,
            "text": text
        })

    return documents

In [6]:
documents = load_pdfs(PDF_DIR)

print("Documents loaded:", len(documents))

Documents loaded: 5


In [7]:
for doc in documents:
    print("=" * 70)
    print(doc["source"])
    print("=" * 70)
    print(doc["text"][:500])
    print()

Work_From_Home_Policy.pdf
BlueOrbit Solutions Pvt. Ltd. — Internal Use Only
Page 1
 BlueOrbit Solutions Pvt. Ltd.
 WORK FROM HOME POLICY
 Document ID: BO-WFH-001    |    Version: 2.0    |    Effective Date: 01-Apr-2026
Document Status: Approved    Owner: Human Resources / Corporate Operations

BlueOrbit Solutions Pvt. Ltd. — Internal Use Only
Page 2
1. Purpose and Scope
This policy defines the conditions under which eligible BlueOrbit employees may work remotely.
WFH availability is subject to role suitability, team requ

Employee_Handbook.pdf
BlueOrbit Solutions Pvt. Ltd. — Internal Use Only
Page 1
 BlueOrbit Solutions Pvt. Ltd.
 EMPLOYEE HANDBOOK
 Document ID: BO-HR-001    |    Version: 2.0    |    Effective Date: 01-Apr-2026
Document Status: Approved    Owner: Human Resources / Corporate Operations

BlueOrbit Solutions Pvt. Ltd. — Internal Use Only
Page 2
1. Purpose and Scope
BlueOrbit Solutions Pvt. Ltd. is a fictional technology services company. This handbook defines
common workp

## 5. Text Cleaning

PDF extraction can introduce unnecessary whitespace and line breaks.

We normalize the extracted text before creating chunks.

The goal is not to aggressively modify the document,
but simply to make the text cleaner and easier to process.

In [8]:
import re


def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [10]:
for doc in documents:
    doc["text"] = clean_text(doc["text"])

print(documents[0]["text"][:1000])

BlueOrbit Solutions Pvt. Ltd. — Internal Use Only Page 1 BlueOrbit Solutions Pvt. Ltd. WORK FROM HOME POLICY Document ID: BO-WFH-001 | Version: 2.0 | Effective Date: 01-Apr-2026 Document Status: Approved Owner: Human Resources / Corporate Operations BlueOrbit Solutions Pvt. Ltd. — Internal Use Only Page 2 1. Purpose and Scope This policy defines the conditions under which eligible BlueOrbit employees may work remotely. WFH availability is subject to role suitability, team requirements, performance, security controls, and manager approval. 2. WFH Eligibility Employees in roles suitable for remote work may request WFH. Roles requiring physical presence, specialized equipment, customer-site work, or other operational coverage may have different arrangements. Eligibility does not create an automatic right to work remotely on any specific day. 3. Monthly WFH Allowance Standard allowance: up to 8 WFH days per calendar month for eligible employees. Unused WFH days do not carry forward to the 

## 6. Text Chunking

Large documents cannot be sent to the embedding model as one huge block.

Therefore, we divide each document into smaller pieces called **chunks**.

For example:

Document
↓
Chunk 1
Chunk 2
Chunk 3
Chunk 4
...

When a user asks a question, we search these chunks to find the
most relevant pieces of information.

### Chunk Configuration

- Chunk size: 800 characters
- Chunk overlap: 150 characters

The overlap helps preserve context between neighboring chunks.

In [11]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150


def create_chunks(documents):
    chunks = []

    for doc in documents:
        text = doc["text"]

        start = 0

        while start < len(text):
            end = start + CHUNK_SIZE

            chunk_text = text[start:end]

            chunks.append({
                "text": chunk_text,
                "source": doc["source"]
            })

            start += CHUNK_SIZE - CHUNK_OVERLAP

    return chunks

In [12]:
chunks = create_chunks(documents)

print("Total chunks:", len(chunks))

Total chunks: 34


In [13]:
print(chunks[0]["source"])
print()
print(chunks[0]["text"])

Work_From_Home_Policy.pdf

BlueOrbit Solutions Pvt. Ltd. — Internal Use Only Page 1 BlueOrbit Solutions Pvt. Ltd. WORK FROM HOME POLICY Document ID: BO-WFH-001 | Version: 2.0 | Effective Date: 01-Apr-2026 Document Status: Approved Owner: Human Resources / Corporate Operations BlueOrbit Solutions Pvt. Ltd. — Internal Use Only Page 2 1. Purpose and Scope This policy defines the conditions under which eligible BlueOrbit employees may work remotely. WFH availability is subject to role suitability, team requirements, performance, security controls, and manager approval. 2. WFH Eligibility Employees in roles suitable for remote work may request WFH. Roles requiring physical presence, specialized equipment, customer-site work, or other operational coverage may have different arrangements. Eligibility does not create an aut


## 7. Generate Embeddings

An embedding converts text into a numerical vector.

For example:

"What is the WFH policy?"

is converted into something like:

[0.021, -0.183, 0.442, ...]

The vector represents the semantic meaning of the text.

We generate an embedding for every document chunk.

Later, we can compare the embedding of a user's question
with the embeddings of our chunks to find semantically similar content.

In [14]:
EMBEDDING_MODEL = "gemini-embedding-001"


def generate_embedding(text):
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text
    )

    return np.array(response.embeddings[0].values, dtype="float32")

In [15]:
test_embedding = generate_embedding("What is the work from home policy?")

print("Embedding dimension:", len(test_embedding))

Embedding dimension: 3072


## 8. Embed All Chunks

Now we generate an embedding for every chunk.

The resulting vectors will be stored in a NumPy array
and then inserted into the FAISS vector index.

In [17]:
embeddings = []

for chunk in chunks:
    embedding = generate_embedding(chunk["text"])
    embeddings.append(embedding)

embeddings = np.array(embeddings, dtype="float32")

print("Embedding matrix shape:", embeddings.shape)

Embedding matrix shape: (34, 3072)


## 9. Create FAISS Vector Index

FAISS is a library for efficient similarity search.

We store the document embeddings inside a FAISS index.

When a user asks a question:

1. Convert the question into an embedding.
2. Search FAISS.
3. Find the most similar document chunks.
4. Pass those chunks to the LLM.

We use inner-product similarity with normalized vectors.
This is equivalent to cosine similarity for normalized embeddings.